# Notebook 21 — TrOCR Fine-Tuning WITH Data Augmentation

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import time, random
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE}")

DATA_ROOT = Path("../data/pharmacy_lk")
TRAIN_CSV = DATA_ROOT/"splits/train.csv"; VAL_CSV = DATA_ROOT/"splits/val.csv"; TEST_CSV = DATA_ROOT/"splits/test.csv"
LABEL_COL = "medicine_name"; IMG_COL = "image_filename"
PROC = "microsoft/trocr-small-handwritten"
CKPT_DIR = Path("../checkpoints/trocr_augmented"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
BATCH = 8; EPOCHS = 60; LR = 5e-5; MAXLEN = 24; PATIENCE = 8

Using device: mps


## 1. Augmentation (TRAIN only) — moderate & realistic

In [2]:
def augment(img):
    """Apply mild, realistic augmentations that mimic prescription-photo variation."""
    # small rotation (handwriting slant / camera tilt)
    if random.random() < 0.6:
        img = img.rotate(random.uniform(-4, 4), resample=Image.BILINEAR, fillcolor=(255,255,255))
    # brightness (lighting variation)
    if random.random() < 0.5:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
    # contrast
    if random.random() < 0.5:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))
    # mild blur (focus/scan softness)
    if random.random() < 0.25:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 0.8)))
    # mild sharpness boost sometimes
    if random.random() < 0.2:
        img = ImageEnhance.Sharpness(img).enhance(random.uniform(1.0, 1.6))
    return img

## 2. Dataset

In [3]:
class TrOCRData(Dataset):
    def __init__(self, csv, processor, train=False):
        self.df = pd.read_csv(csv).dropna(subset=[LABEL_COL]).reset_index(drop=True)
        self.df[LABEL_COL] = self.df[LABEL_COL].astype(str).str.strip().str.lower()
        self.processor = processor; self.train = train; self.img_dir = DATA_ROOT/"images"
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(self.img_dir/str(r[IMG_COL])).convert("RGB")
        if self.train:
            img = augment(img)
        pv = self.processor(img, return_tensors="pt").pixel_values.squeeze(0)
        labels = self.processor.tokenizer(r[LABEL_COL], padding="max_length",
                  max_length=MAXLEN, truncation=True).input_ids
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels]
        return {"pixel_values": pv, "labels": torch.tensor(labels),
                "text": r[LABEL_COL], "fname": str(r[IMG_COL])}

def collate(b):
    return {"pixel_values": torch.stack([x["pixel_values"] for x in b]),
            "labels": torch.stack([x["labels"] for x in b]),
            "text": [x["text"] for x in b], "fname": [x["fname"] for x in b]}

## 3. Metrics + load model

In [4]:
def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a,1):
        cur=[i]
        for j, cb in enumerate(b,1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca!=cb)))
        prev=cur
    return prev[-1]
def metrics(preds, refs):
    tot=sum(edit_distance(p,r) for p,r in zip(preds,refs)); chars=sum(len(r) for r in refs)
    em=sum(p==r for p,r in zip(preds,refs))
    return {"CER":tot/max(chars,1), "EM":em/len(refs), "n":len(refs)}

processor = TrOCRProcessor.from_pretrained(PROC)
model = VisionEncoderDecoderModel.from_pretrained(PROC).to(DEVICE)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
print(f"TrOCR loaded | {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

train_dl = DataLoader(TrOCRData(TRAIN_CSV, processor, train=True), BATCH, shuffle=True, num_workers=0, collate_fn=collate)
val_dl   = DataLoader(TrOCRData(VAL_CSV, processor), BATCH, shuffle=False, num_workers=0, collate_fn=collate)
test_dl  = DataLoader(TrOCRData(TEST_CSV, processor), BATCH, shuffle=False, num_workers=0, collate_fn=collate)

@torch.no_grad()
def evaluate(loader):
    model.eval(); preds, refs, files = [], [], []
    for b in loader:
        gen = model.generate(b["pixel_values"].to(DEVICE), max_new_tokens=MAXLEN)
        preds += [o.strip().lower() for o in processor.batch_decode(gen, skip_special_tokens=True)]
        refs += [t.lower() for t in b["text"]]; files += b["fname"]
    return metrics(preds, refs), preds, refs, files

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TrOCR loaded | 61.6M params


## 4. Train with early stopping

In [5]:
opt = torch.optim.AdamW(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
best, no_imp = float("inf"), 0
for epoch in range(1, EPOCHS+1):
    model.train(); t0, run = time.time(), 0.0
    for k, b in enumerate(train_dl):
        out = model(pixel_values=b["pixel_values"].to(DEVICE), labels=b["labels"].to(DEVICE))
        loss = out.loss; opt.zero_grad(); loss.backward(); opt.step(); run += loss.item()
        if k % 20 == 0: print(f"  ep{epoch} {k}/{len(train_dl)} loss {loss.item():.3f}", end="\r")
    vm,_,_,_ = evaluate(val_dl); sched.step(vm["CER"])
    print(f"\nepoch {epoch} | train loss {run/len(train_dl):.4f} | val CER {vm['CER']:.4f} | "
          f"val EM {vm['EM']:.4f} | {(time.time()-t0)/60:.1f}m")
    if vm["CER"] < best:
        best, no_imp = vm["CER"], 0; model.save_pretrained(CKPT_DIR/"best")
        print(f"  saved best (val CER {best:.4f})")
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f"early stop epoch {epoch} (best val CER {best:.4f})"); break

  ep1 460/461 loss 0.956
epoch 1 | train loss 2.2109 | val CER 0.2775 | val EM 0.4101 | 3.0m


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.2775)
  ep2 460/461 loss 0.540
epoch 2 | train loss 1.0910 | val CER 0.2667 | val EM 0.4506 | 2.9m


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.2667)
  ep3 460/461 loss 1.101
epoch 3 | train loss 0.8225 | val CER 0.2385 | val EM 0.5203 | 2.9m


...

  saved best (val CER 0.1547)
  ep25 460/461 loss 0.009
epoch 25 | train loss 0.0197 | val CER 0.1652 | val EM 0.6595 | 2.8m
  ep26 460/461 loss 0.008
epoch 26 | train loss 0.0173 | val CER 0.1560 | val EM 0.6620 | 2.8m
  ep27 460/461 loss 0.006
epoch 27 | train loss 0.0156 | val CER 0.1571 | val EM 0.6595 | 2.8m
  ep28 460/461 loss 0.005
epoch 28 | train loss 0.0121 | val CER 0.1505 | val EM 0.6759 | 2.8m


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.1505)
  ep29 460/461 loss 0.007
epoch 29 | train loss 0.0108 | val CER 0.1538 | val EM 0.6734 | 2.8m
  ep30 460/461 loss 0.004
epoch 30 | train loss 0.0093 | val CER 0.1586 | val EM 0.6722 | 2.8m
  ep31 460/461 loss 0.026
epoch 31 | train loss 0.0078 | val CER 0.1505 | val EM 0.6696 | 2.8m
  ep32 460/461 loss 0.002
epoch 32 | train loss 0.0075 | val CER 0.1523 | val EM 0.6722 | 2.8m
  ep33 460/461 loss 0.002
epoch 33 | train loss 0.0066 | val CER 0.1479 | val EM 0.6734 | 2.8m


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.1479)
  ep34 460/461 loss 0.001
epoch 34 | train loss 0.0074 | val CER 0.1518 | val EM 0.6734 | 2.8m
  ep35 460/461 loss 0.005
epoch 35 | train loss 0.0069 | val CER 0.1520 | val EM 0.6759 | 2.8m
  ep36 460/461 loss 0.005
epoch 36 | train loss 0.0064 | val CER 0.1498 | val EM 0.6759 | 2.8m
  ep37 460/461 loss 0.016
epoch 37 | train loss 0.0056 | val CER 0.1522 | val EM 0.6759 | 2.8m
  ep38 460/461 loss 0.007
epoch 38 | train loss 0.0066 | val CER 0.1525 | val EM 0.6772 | 2.8m
  ep39 460/461 loss 0.003
epoch 39 | train loss 0.0055 | val CER 0.1527 | val EM 0.6709 | 2.8m
  ep40 460/461 loss 0.004
epoch 40 | train loss 0.0050 | val CER 0.1536 | val EM 0.6722 | 2.8m
  ep41 460/461 loss 0.004
epoch 41 | train loss 0.0050 | val CER 0.1547 | val EM 0.6709 | 2.8m
early stop epoch 41 (best val CER 0.1479)


## 5. Test + compare to un-augmented TrOCR

In [6]:
model = VisionEncoderDecoderModel.from_pretrained(CKPT_DIR/"best").to(DEVICE).eval()
tm, preds, refs, files = evaluate(test_dl)
print(f"\nTrOCR + augmentation — TEST: CER {tm['CER']:.4f} | EM {tm['EM']:.4f}")
print(f"""
COMPARISON:
  TrOCR (no augmentation)   : CER 0.216 | EM 0.569   (from Notebook 11)
  TrOCR + augmentation      : CER {tm['CER']:.3f} | EM {tm['EM']:.3f}   (this run)

If EM rises meaningfully -> augmentation improved recognition (report as a quality gain).
If ~same or lower -> honest finding: augmentation did not help on this data (the dataset's
real-world variation may already be high, or moderate aug was insufficient/too strong).
""")
pd.DataFrame([{"model":"TrOCR+aug", **tm}]).to_csv(CKPT_DIR/"aug_results.csv", index=False)

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]


TrOCR + augmentation — TEST: CER 0.1762 | EM 0.6549

COMPARISON:
  TrOCR (no augmentation)   : CER 0.216 | EM 0.569   (from Notebook 11)
  TrOCR + augmentation      : CER 0.176 | EM 0.655   (this run)

If EM rises meaningfully -> augmentation improved recognition (report as a quality gain).
If ~same or lower -> honest finding: augmentation did not help on this data (the dataset's
real-world variation may already be high, or moderate aug was insufficient/too strong).

